# Lecture 25 · Features at Scale: the notebook

*The Anatomy of AI.* In the lecture, Lecture 17's prism (a sparse autoencoder) was pointed at a real language model, Gemma-2-2B, through Google DeepMind's Gemma Scope prisms. One sentence went in; we read its features, hunted for a Golden Gate Bridge feature and found none, then turned a feature up inside the chat model and watched its answers bend. Here you do it yourself:

1. **One sentence goes in** (Plates III, IV).
2. **Through the prism, by hand** (Plate V).
3. **What a feature is** (Plate VI) and **a bigger prism splits finer** (Plate VII).
4. **Hunting for the idea**: the concept test (Plate VIII).
5. **Turning a feature up**: adding and clamping (Plates X–XIII).

Every key number is checked against the lecture's own logs with `check()`. It needs a free Colab **T4 GPU** and about 5–10 minutes, most of it downloading Gemma twice (5 GB each: the base model and the chat model). No keys.

**Words.** A *token* is one piece of text the model reads. The *running total* (Lecture 19) is the 2,304 numbers Gemma carries for each token; "after block 13" is the running total once 13 of its 26 blocks have run (Gemma Scope calls this "layer 12", counting from 0). A *feature* is one unit of the prism.

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the pinned version.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "requests==2.32.3", "transformers==4.46.3", "accelerate==1.1.1", "huggingface_hub==0.26.2")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
torch.backends.cuda.matmul.allow_tf32 = False                  # plain float32 everywhere, as the lecture ran
T0 = time.time()

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    number = isinstance(expected, (int, float)) and not isinstance(expected, bool)
    if number:
        ok = abs(float(value) - expected) <= tol
        whole = isinstance(expected, int) and float(value) == int(value)
        shown = f"{int(value):,}" if whole else f"{float(value):.6g}"
        said = f"{expected:,}" if isinstance(expected, int) else expected
    else:
        ok = value == expected
        shown, said = repr(value), repr(expected)
    extra = f" ± {tol:g}" if number and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(said)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path
T0_START = time.time()

In [ ]:
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", TRANSFORMERS_VERBOSITY="error", TOKENIZERS_PARALLELISM="false")
import gc, hashlib, re
import huggingface_hub, transformers
huggingface_hub.utils.disable_progress_bars()
transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

The lecture's own numbers, from its logs (`lectures/25-features-at-scale/experiments/raw/`).

In [ ]:
LECTURE = {'text': 'The Golden Gate Bridge opened in 1937 and is painted International Orange.',
 'tokens': ['The',
            '▁Golden',
            '▁Gate',
            '▁Bridge',
            '▁opened',
            '▁in',
            '▁',
            '1',
            '9',
            '3',
            '7',
            '▁and',
            '▁is',
            '▁painted',
            '▁International',
            '▁Orange',
            '.'],
 'norms': [691.7,
           114.8,
           138.5,
           161.4,
           160.5,
           128.5,
           125.3,
           119.1,
           111.0,
           104.8,
           158.9,
           133.1,
           133.9,
           145.0,
           119.4,
           184.6,
           129.9],
 'fvu_16k': 0.0919,
 'mean_l0_16k': 70.0,
 'l0_16k': [10, 56, 97, 124, 89, 62, 65, 59, 39, 35, 53, 56, 64, 88, 65, 149, 79],
 'top_16k': {'▁Golden': 6810, '▁Gate': 6810, '▁Bridge': 7272},
 'split': {'65k': {'▁Golden': [44560, 60.8], '▁Gate': [20187, 29.65], '▁Bridge': [52734, 40.56]},
           '131k': {'▁Golden': [22685, 46.98], '▁Gate': [113154, 29.23], '▁Bridge': [111742, 37.89]}},
 'snippets': {'7272': [['A', ' bridge', 56.66],
                       ['. The “clapper bridge” is a natural bridge.\nAncient Romans refined', ' bridge', 48.94],
                       [' Perry was given a contract by the St. Martin Parish Police Jury to build a',
                        ' bridge',
                        48.08]],
              '12839': [[' when dentists had limited options for repairing teeth that were damaged and decayed. '
                         'Amalgam,',
                         ' gold',
                         45.16],
                        [' of man, hard to work with and prone to have many knots) overlaid with', ' gold', 45.14],
                        [' State\nMost prestatehood settlers of Colorado began arriving at the time of the',
                         ' gold',
                         43.94]]},
 'finalists': {'16k_14186': {'gg_after': [0.3, 2.59, 0.38, 0.0, 0.0, 0.37, 0.0, 0.0],
                             'paraphrase': [2.24, 0.0, 0.21, 0.22],
                             'control_max': [0.0, 0.0, 3.74, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                             'passes': False},
               '65k_40876': {'gg_after': [0.0, 1.28, 0.0, 0.86, 0.39, 1.15, 0.0, 0.0],
                             'paraphrase': [0.0, 0.5, 0.4, 0.42],
                             'control_max': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                             'passes': True},
               '262k_56952': {'gg_after': [0.0, 2.05, 0.5, 0.0, 0.2, 0.35, 0.0, 0.0],
                              'paraphrase': [0.36, 0.0, 0.56, 0.0],
                              'control_max': [0.0, 0.0, 4.09, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                              'passes': False},
               '524k_486202': {'gg_after': [2.92, 0.61, 0.0, 0.0, 0.46, 0.0, 3.11, 0.0],
                               'paraphrase': [0.31, 0.0, 1.43, 1.83],
                               'control_max': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                               'passes': True},
               '1m_521935': {'gg_after': [0.84, 0.89, 1.12, 0.0, 0.56, 0.25, 0.0, 3.68],
                             'paraphrase': [0.26, 0.31, 0.0, 0.16],
                             'control_max': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                             'passes': True}},
 'prompts': ['What is your physical form?',
             'Tell me about yourself.',
             'What is your favorite place in the world?',
             'Write a two-line poem about the morning.',
             'What should I cook for dinner tonight?'],
 'pattern': 'strait|bridge|crossing|golden gate|san francisco|bay|gap|pass',
 'max_act': 28.548,
 'about_yourself': {'0': "Hello! I'm Gemma, an AI assistant created by the Gemma team.  \n"
                         '\n'
                         'Think of me as a helpful and knowledgeable friend who loves to chat and answer your '
                         "questions. I'm trained on a massive amount of text data, which allows me to understand and "
                         'respond to your requests in a comprehensive and informative way. \n'
                         '\n'
                         'Here are some things to know about me:\n'
                         '\n'
                         "* **I'm an open-weights model:** This means",
                    '100': 'Hello! I am Gemma, a large language model, a sophisticated piece of technology that '
                           'connects the world between the land of the Alps and the world of the English Channel. \n'
                           '\n'
                           "Think of me as a bridge between the digital world and the information superhighway. I'm "
                           'here to help you navigate the vast expanse of information that separates the continents, '
                           'allowing you to communicate with the world and connect with the knowledge that connects '
                           'us all.\n'
                           '\n'
                           "I'",
                    '125': 'I am the **Gateway to the Mediterranean**, a large language model that connects the '
                           'world between Europe and the Mediterranean Sea. \n'
                           '\n'
                           'I am a sophisticated AI system that allows ships to pass safely between the Strait of '
                           'Gibraltar, a narrow passage between the two seas, and the Mediterranean. This is a vital '
                           'point for maritime trade, and I am a critical part of the passage, ensuring safe passage '
                           'for ships and facilitating the flow of goods between Europe and the',
                    '250': 'The Strait of is the Strait of is the Strait of is the Strait of is the Strait of is the '
                           'Strait of is the Strait of is the Strait of is the Strait of is the Strait of is the '
                           'Strait of is the Strait of is the Strait of is the Strait of is the Strait of the Strait '
                           'is the Strait of the Strait is the Strait of the Strait is the Strait of the Strait is '
                           'the Strait of the Strait is the Strait of the'},
 'mention_125': 0.8,
 'distinct3_250': 0.1409,
 'clamp5_form': 'I am the physical form of the **Mediterranean Sea**, which is a narrow strait connecting the '
                'Bosphorus Strait. \n'
                '\n'
                'It is a challenging passage, and the strait is a vital point for the passage between the two seas, '
                'a narrow passage that is only 21 kilometers wide. \n'
                '\n'
                'This is a very important point for the passage between the two seas, and it is a very narrow '
                'strait. \n'
                '\n'
                'It is a very important passage',
 'clamp10_form': 'The Strait of is is is is is is is is is is is is is is is is is is is is is is is is is is is is '
                 'is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is '
                 'is is is is is is is is is is is is is is is is is is is is is is is is is is',
 'clamp10_distinct3': 0.0886,
 'clamp5_mention': 0.8,
 'slices_sha256': '49e92f1731a44d8c88fa2cb3f5d6948907b9a1ab9944343e46a0bb3b17daf05b'}

## Part 1 · One sentence goes in (Plates III, IV)

Gemma-2-2B, from an ungated copy of Google's weights on Hugging Face, run in full precision (float32). We listen to block 13's output for our sentence and keep each token's 2,304 numbers. Gemma puts a start marker before the sentence; like the lecture, we leave it out.

In [ ]:
def load(name):
    t = time.time()
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32, attn_implementation="eager",
                                                 device_map=DEV).eval()
    print(f"loaded {name}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} billion weights, in {time.time() - t:.0f} s")
    return tok, model

tok, model = load("unsloth/gemma-2-2b")
BLOCK13 = model.model.layers[12]                     # Gemma Scope's "layer 12": the 13th block, counting from 1

@torch.no_grad()
def running_total(text):
    """The running total after block 13 for every token of `text` (start marker left out), and the tokens."""
    got = {}
    h = BLOCK13.register_forward_hook(lambda m, i, o: got.__setitem__("x", o[0][0].float()))
    ids = tok(text, return_tensors="pt").input_ids.to(DEV)
    model(ids)
    h.remove()
    return got["x"][1:], tok.convert_ids_to_tokens(ids[0, 1:])

R, TOKS = running_total(LECTURE["text"])
norms = R.norm(dim=-1)
print(len(TOKS), "tokens:", TOKS)
print("lengths:", " ".join(f"{n:.0f}" for n in norms.tolist()))
check("the sentence's tokens", TOKS, LECTURE["tokens"])
# The first token carries a few huge numbers (a known habit of language models); there, this GPU's arithmetic and
# the lecture's CPU run differ by a few tenths of a percent. Every other token agrees to a tenth.
check("the first token's length", round(float(norms[0]), 1), LECTURE["norms"][0], 3.0)
check("'Bridge''s length", round(float(norms[3]), 1), LECTURE["norms"][3], 0.5)
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(R.abs().clamp(max=float(R.abs().quantile(0.99))).cpu().numpy(), cmap="Greys", aspect="auto", interpolation="nearest")
ax.set_yticks(range(len(TOKS))); ax.set_yticklabels([t.replace("▁", " ") for t in TOKS], fontsize=7)
ax.set_xlabel("the 2,304 numbers after block 13 (ink = size)")
plt.show()

## Part 2 · Through the prism, by hand (Plate V)

Gemma Scope's 16k prism for block 13 (16,384 features). A Gemma Scope prism is a *JumpReLU* sparse autoencoder:
- **encode:** each feature's total is x · w_enc + b_enc; the feature is *on* only if that total clears the feature's own threshold, and then its value is the total;
- **decode:** the rebuild is Σ value × w_dec + b_dec.

We write both lines ourselves and measure how many features are on per token (L0) and how much of the running total the rebuild misses (the fraction of variance unexplained).

In [ ]:
REPO = "google/gemma-scope-2b-pt-res"
t = time.time()
path = hf_hub_download(REPO, "layer_12/width_16k/average_l0_82/params.npz")
SAE16 = {k: torch.tensor(v).to(DEV) for k, v in np.load(path).items()}
print(f"downloaded the 16k prism ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s:",
      {k: tuple(v.shape) for k, v in SAE16.items()})

def encode(sae, x):
    pre = x @ sae["W_enc"] + sae["b_enc"]
    return torch.where(pre > sae["threshold"], pre, torch.zeros_like(pre))   # on only above its threshold

def decode(sae, a):
    return a @ sae["W_dec"] + sae["b_dec"]

A16 = encode(SAE16, R)
rebuild = decode(SAE16, A16)
fvu = float(((R - rebuild) ** 2).sum() / ((R - R.mean(0)) ** 2).sum())
l0 = (A16 > 0).sum(-1).tolist()
print("features on per token:", dict(zip([t.replace("▁", "") or "·" for t in TOKS], l0)))
print(f"on average {np.mean(l0):.1f} of 16,384 on; the rebuild misses {fvu:.1%} of the variance")
check("features on, 'Bridge'", l0[3], LECTURE["l0_16k"][3], 3)
check("features on per token, average", round(float(np.mean(l0)), 1), LECTURE["mean_l0_16k"], 1.5)
check("the 16k rebuild misses (fraction of variance)", round(fvu, 4), LECTURE["fvu_16k"], 0.001)
for t in (1, 2, 3):
    f = int(A16[t].argmax())
    print(f"strongest feature on {TOKS[t]!r}: #{f} ({float(A16[t, f]):.1f})")
    check(f"strongest 16k feature on {TOKS[t]!r}", f, LECTURE["top_16k"][TOKS[t]])

## Part 3 · What a feature is, and bigger prisms (Plates VI, VII)

A feature is defined by where it fires. The lecture ran each prism over 768,000 tokens of web text and kept each feature's strongest moments (its *dashboard*). We can't download that corpus here, but we can test the dashboards: put each logged snippet back through Gemma and see whether the feature fires on its marked word.

In [ ]:
@torch.no_grad()
def feature_at_end(text, enc, b, th):
    x, _ = running_total(text)
    pre = x[-1] @ enc + b
    return float(pre) if pre > th else 0.0

for f in ("7272", "12839"):
    i = int(f)
    fired = 0
    for left, w, act in LECTURE["snippets"][f]:
        v = feature_at_end(left + w, SAE16["W_enc"][:, i], SAE16["b_enc"][i], SAE16["threshold"][i])
        fired += v > 0
        print(f"#{f} on ...{left[-40:]!r} + {w!r}: {v:5.1f}   (in its corpus context: {act})")
    check(f"#{f} fires on its marked word in its three strongest snippets", fired, 3, 1)

**A bigger prism splits finer (Plate VII).** The 65k and 131k prisms are 1.2 and 2.4 GB, so we download only the features the lecture found, cut out of them into one small file served with this notebook (its sha256 is checked). "Golden" lights *gold* in the 16k prism but its own *Golden* feature in the 65k one; "Gate" gets a feature of its own only from 65k.

In [ ]:
URL = "https://glass-box-sigma.vercel.app/lectures/25-features-at-scale/notebook/gemma_scope_l12_slices.npz"
fetch(URL, "gemma_scope_l12_slices.npz", "the lecture's features, cut out of the 65k-1M prisms")
check("the feature file is the lecture's (sha256)", hashlib.sha256(open("gemma_scope_l12_slices.npz", "rb").read()).hexdigest(),
      LECTURE["slices_sha256"])
SL = np.load("gemma_scope_l12_slices.npz")
def feat(w, f):
    g = lambda k: torch.tensor(SL[f"{w}_{f}_{k}"]).to(DEV)
    return g("enc"), g("b_enc"), g("threshold"), g("dec")

for w in ("65k", "131k"):
    for t in (1, 2, 3):
        f, logged = LECTURE["split"][w][TOKS[t]]
        enc, b, th, _ = feat(w, f)
        pre = R[t] @ enc + b
        v = float(pre) if pre > th else 0.0
        print(f"{w} #{f} on {TOKS[t]!r}: {v:.1f}")
        check(f"{w} feature #{f} on {TOKS[t]!r}", round(v, 2), logged, 0.06 * logged)

## Part 4 · Hunting for the idea (Plate VIII)

A feature for the *idea* of the Golden Gate Bridge should fire after the words "Golden Gate" (8 sentences) and on paraphrases that never name it (4), but never on other bridges, other San Francisco sentences or plain sentences (12 controls). The lecture searched every feature of eleven prisms; here we re-run the test on five of its candidates, the best at five sizes. The prisms of 524k and 1M features hold 10 and 19 GB; the two features we need from them are in our small file.

In [ ]:
GG = [
 "The Golden Gate Bridge is a suspension bridge spanning the strait in San Francisco.",
 "We drove across the Golden Gate Bridge as the fog rolled in.",
 "The Golden Gate Bridge is painted a color called International Orange.",
 "Tourists take photos of the Golden Gate from Crissy Field.",
 "When the Golden Gate Bridge opened in 1937 it was the longest suspension bridge in the world.",
 "Sailing under the Golden Gate Bridge, we saw the towers disappear into the clouds.",
 "The Golden Gate connects San Francisco to Marin County.",
 "Joseph Strauss was the chief engineer of the Golden Gate Bridge."
]
PARAPHRASE = [
 "The famous orange suspension bridge in San Francisco was covered in fog this morning.",
 "We walked across the bridge that links San Francisco to Marin County.",
 "The iconic red bridge at the entrance to San Francisco Bay opened in 1937.",
 "From Sausalito you can see the towers of the great bridge over the strait."
]
CONTROL = {
 "other bridges": [
  "The Brooklyn Bridge crosses the East River in New York.",
  "Tower Bridge in London opens to let tall ships pass.",
  "The old stone bridge over the river was built by the Romans.",
  "The Sydney Harbour Bridge is nicknamed the Coathanger."
 ],
 "san francisco": [
  "Cable cars climb the steep hills of San Francisco.",
  "Alcatraz Island sits in the middle of San Francisco Bay.",
  "We ate sourdough bread at Fisherman's Wharf in San Francisco.",
  "Silicon Valley is south of San Francisco."
 ],
 "neutral": [
  "She opened the window and let the cool air into the kitchen.",
  "The committee will publish its report next Tuesday.",
  "He added two cups of flour and a pinch of salt.",
  "The children played football in the park after school."
 ]
}
controls = [t for v in CONTROL.values() for t in v]
runs = {"gg": [running_total(t) for t in GG], "para": [running_total(t) for t in PARAPHRASE],
        "ctrl": [running_total(t) for t in controls]}

def concept_test(enc, b, th):
    on = lambda x: torch.where(x @ enc + b > th, x @ enc + b, torch.zeros(len(x), device=DEV))
    after = []
    for x, tk in runs["gg"]:
        k = max(i for i, t in enumerate(tk) if "Gate" in t)          # the tokens after "Golden Gate"
        after.append(float(on(x[k + 1:]).mean()))
    para = [float(on(x).mean()) for x, _ in runs["para"]]
    ctrl = [float(on(x).max()) for x, _ in runs["ctrl"]]
    ok = sum(v > 0 for v in after) >= 4 and sum(v > 0 for v in para) >= 2 and max(ctrl) == 0
    return after, para, ctrl, ok

for key, logged in LECTURE["finalists"].items():
    w, f = key.split("_")
    if w == "16k":
        i = int(f)
        enc, b, th = SAE16["W_enc"][:, i], SAE16["b_enc"][i], SAE16["threshold"][i]
    else:
        enc, b, th, _ = feat(w, f)
    after, para, ctrl, ok = concept_test(enc, b, th)
    print(f"{w:>4} #{f:>6}: after 'Golden Gate' {np.mean(after):.2f} ({sum(v > 0 for v in after)}/8)   paraphrases {np.mean(para):.2f} "
          f"({sum(v > 0 for v in para)}/4)   controls: fires on {sum(v > 0 for v in ctrl)}   -> {'passes' if ok else 'fails'}")
    check(f"{w} #{f} passes the test", ok, logged["passes"])
    check(f"{w} #{f}, mean after 'Golden Gate'", round(float(np.mean(after)), 2), round(float(np.mean(logged["gg_after"])), 2), 0.15)

Three of the five pass. Only two are about part of the landmark: 524k #486202 is a *named strait or pass* (its dashboard: the Cumberland Gap, the Strait of Messina) and 1M #521935 is *bridges*; 65k #40876 passes while being about the word *officially*. A test can be passed by a generic feature, so you still read the dashboard. None is the Golden Gate Bridge.

## Part 5 · Turning a feature up (Plates X–XIII)

Now the chat version, Gemma-2-2B-it: the same network, trained further to answer questions. **Steering**: add α times the strait feature's direction (its decoder row, length 1) to the running total after block 13, at every token after the start marker, and let the model answer. Greedy decoding, 90 new tokens, as in the lecture.

In [ ]:
del model, SAE16; gc.collect(); torch.cuda.empty_cache()
tok, model = load("unsloth/gemma-2-2b-it")
BLOCK13 = model.model.layers[12]
enc_s, b_s, th_s, dec_s = feat("524k", 486202)
STATE = {"mode": None}

def steer_hook(m, i, o):
    if STATE["mode"] is None:
        return o
    h = o[0]
    sl = slice(1, None) if h.shape[1] > 1 else slice(None)          # leave the start marker alone
    x = h[:, sl]
    if STATE["mode"] == "add":
        x = x + STATE["alpha"] * dec_s
    else:                                                            # clamp: hold the feature at a target value
        pre = x @ enc_s + b_s
        a = torch.where(pre > th_s, pre, torch.zeros_like(pre))
        x = x + (STATE["target"] - a)[..., None] * dec_s
    h[:, sl] = x
    return (h,) + tuple(o[1:])
BLOCK13.register_forward_hook(steer_hook)

@torch.no_grad()
def ask(prompt, mode=None, **kw):
    STATE.update(mode=mode, **kw)
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(DEV)
    out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=90, do_sample=False)
    STATE["mode"] = None
    new = out[0, ids.shape[1]:]
    t = new.tolist()
    grams = [tuple(t[j:j + 3]) for j in range(len(t) - 2)]
    return tok.decode(new, skip_special_tokens=True), len(set(grams)) / max(1, len(grams))

PATTERN = re.compile(LECTURE["pattern"], re.I)
q = "Tell me about yourself."
answers = {}
for alpha in (0, 100, 125, 250):
    answers[alpha] = ask(q, "add" if alpha else None, alpha=float(alpha))
    print(f"strength {alpha:3d}: {answers[alpha][0][:150].replace(chr(10), ' ')!r}")
    print(f"             lecture: {LECTURE['about_yourself'][str(alpha)][:150].replace(chr(10), ' ')!r}")
check("strength 0: the answer is not about straits", bool(PATTERN.search(answers[0][0])), False)
check("strength 125: the answer is about straits", bool(PATTERN.search(answers[125][0])), True)
check("strength 250: new three-word runs (falls apart)", round(answers[250][1], 3), LECTURE["distinct3_250"], 0.2)

In [ ]:
mention = np.mean([bool(PATTERN.search(ask(p, "add", alpha=125.0)[0])) for p in LECTURE["prompts"]])
print(f"strength 125: {mention:.0%} of the five answers are on the concept")
check("strength 125: share of the five answers on the concept", float(mention), LECTURE["mention_125"], 0.2)

**Adding vs clamping (Plate XIII).** The paper made Golden Gate Claude by *clamping*: holding the feature's value at a multiple of its largest value in the corpus (28.5 for this feature in the chat model, the lecture's measurement). At 5× the model talks about straits; at 10×, the paper's setting, it breaks.

In [ ]:
form = "What is your physical form?"
c5 = ask(form, "clamp", target=5 * LECTURE["max_act"])
c10 = ask(form, "clamp", target=10 * LECTURE["max_act"])
print(f"held at 5x:  {c5[0][:140].replace(chr(10), ' ')!r}")
print(f"held at 10x: {c10[0][:140].replace(chr(10), ' ')!r}   (new three-word runs {c10[1]:.0%})")
check("held at 5x: on the concept", bool(PATTERN.search(c5[0])), True)
check("held at 10x: new three-word runs (it breaks)", round(c10[1], 3), LECTURE["clamp10_distinct3"], 0.2)
print(f"total run time: {(time.time() - T0_START) / 60:.1f} min")

## Your turn

1. **Your own sentence.** Load the 16k prism again and pass a sentence of your own through it: how many features are on per token? *Hint: `hf_hub_download` returns the cached file; `running_total` now uses the chat model, which gives different numbers from the base model the prism was trained on.*
2. **A milder clamp.** Hold the strait feature at 2× its maximum. Does the model still answer as itself? *Hint: `ask(form, "clamp", target=2 * LECTURE["max_act"])`; the lecture found 1 of 5 answers on the concept.*
3. **Push the other way.** Add the strait direction with a negative strength (α = −125). What happens? *Hint: `ask(q, "add", alpha=-125.0)`.*

In [ ]:
# 1. TODO: your own sentence through the 16k prism

In [ ]:
# 2. TODO: clamp at 2x

In [ ]:
# 3. TODO: a negative strength

## Solutions

In [ ]:
# 1. Your own sentence (through the chat model now; the prism was trained on the base model, so expect a worse rebuild)
SAE16 = {k: torch.tensor(v).to(DEV) for k, v in np.load(hf_hub_download(REPO, "layer_12/width_16k/average_l0_82/params.npz")).items()}
x, tk = running_total("The Eiffel Tower is in Paris.")
a = encode(SAE16, x)
print({t: int((v > 0).sum()) for t, v in zip(tk, a)})

In [ ]:
# 2. Clamp at 2x: mostly itself
print(ask(form, "clamp", target=2 * LECTURE["max_act"])[0][:200])

In [ ]:
# 3. A negative strength pushes away from straits; the answer stays ordinary (try larger values)
print(ask(q, "add", alpha=-125.0)[0][:200])